### **Title:** RNN Encoder-Decoder for Statistical Machine Translation

### **Objectives:**
* To implement a Sequence-to-Sequence (Seq2Seq) model using Recurrent Neural Networks (RNNs) for machine translation.
* To preprocess bilingual text data by tokenizing sentences, building vocabularies, and preparing input-output sequences for training.
* To train and evaluate an encoder-decoder network, understanding the roles of embeddings, hidden states, and teacher forcing in sequence learning and translation.

### **Theory:**
The Encoder–Decoder architecture, also known as the Sequence-to-Sequence (Seq2Seq) architecture, is a neural network framework designed for sequence-to-sequence learning tasks such as machine translation, text summarization, and speech recognition. Unlike traditional machine learning models that operate on fixed-size inputs and outputs, Seq2Seq models can process variable-length sequences, making them suitable for tasks such as machine translation, text summarization, chatbot development, speech recognition, and question answering.

The architecture mainly consists of two neural networks:
* Encoder
* Decoder

#### **Working Principle:**

The Seq2Seq model operates through the following stages:

* Tokenize the input sentence into individual words.
* Convert each word into a numerical index and embedding vector.
* The encoder processes the input sequence and generates a context vector.
* The context vector is passed to the decoder.
* The decoder starts with the SOS token and predicts one word at a time.
* Each predicted word is used to generate the next word.
* The process stops when the EOS token is produced, completing the translation.


### **Encoder-Decoder Architecture:**
```text
Source Sentence
       │
       ▼
Tokenization
       │
       ▼
Encoder (Embedding + RNN)
       │
       ▼
Context Vector (Hidden State)
       │
       ▼
Decoder (Embedding + RNN)
       │
       ▼
Predicted Sentence
```

| **Stage**              | **Purpose**                                                                                    |
| ---------------------- | ---------------------------------------------------------------------------------------------- |
| **Input Sentence**     | The original source sentence that will be translated or processed by the model.                |
| **Tokenization**       | Splits the input sentence into individual words or tokens.                                     |
| **Word Indices**       | Converts each token into its corresponding numerical ID using the vocabulary.                  |
| **Embedding Layer**    | Maps each word index to a dense vector representation (embedding).                             |
| **Dense Word Vectors** | Continuous vector representations that capture semantic and syntactic information about words. |
| **Encoder RNN**        | Processes the sequence of word embeddings and encodes the input into a hidden representation.  |
| **Context Vector**     | A fixed-length vector containing the encoded information of the entire input sequence.         |
| **Decoder RNN**        | Uses the context vector to generate the output sequence one word at a time.                    |
| **Output Sentence**    | The final translated or predicted sentence produced by the decoder.                            |



### **Teacher Forcing:**

Teacher Forcing is a training technique used in Seq2Seq models where the decoder receives the correct target word from the training data as the next input instead of its own previous prediction. This helps the model learn faster, improves training accuracy, and reduces error propagation. During testing, teacher forcing is not used because the correct target sequence is unavailable.

Example of Teacher Forcing

Target sentence: SOS → I → am → happy → EOS

With Teacher Forcing: SOS → I → am → happy
(Uses the correct previous word as input.)
Without Teacher Forcing: SOS → I → am → ...
(Uses its own previous prediction as input.)

### **Context Vector:**

The Context Vector is the final hidden state produced by the encoder after processing the entire input sequence. It contains the key information and overall meaning of the input sentence, which is then passed to the decoder to generate the output sequence. A more informative context vector generally results in more accurate translations or predictions.
Example:

Input: I am happy

Encoder → Context Vector → Decoder

The context vector stores the meaning of "I am happy", enabling the decoder to generate the translated sentence, e.g., "Je suis heureux."

### **Word Embeddings:**

Neural networks cannot process words directly. Therefore, each word is first converted into a numerical index and then mapped to a dense vector through an embedding layer. These vectors capture semantic similarities between words, allowing the model to learn more meaningful representations efficiently.
| Word | One-Hot   | Embedding             |
| ---- | --------- | --------------------- |
| Cat  | `[0,1,0]` | `[0.25, -0.81, 0.63]` |
| Dog  | `[1,0,0]` | `[0.27, -0.79, 0.60]` |

The embeddings of cat and dog are similar because they are semantically related.

### **Loss Function:**

The model evaluates its predictions using Negative Log-Likelihood Loss (NLLLoss) by comparing the predicted output sequence with the target sequence. The loss value indicates the prediction error, where a lower loss signifies better model performance and more accurate translations.
Example:

Target: I am happy

Prediction: I am sad

Since the predicted sentence differs from the target sentence, the NLLLoss is high. If the prediction is I am happy, the loss becomes low, indicating better model performance.

### **Applications:**

Sequence-to-Sequence models are widely used in:

* Machine Translation
* Chatbots
* Text Summarization
* Speech Recognition
* Question Answering Systems
* Language Generation
* Conversational AI



### **Why Not Feed One-Hot Vectors Directly into an RNN?**

A common question in Natural Language Processing (NLP) is why an embedding layer is used before an RNN when words can already be represented as one-hot vectors.

Although an RNN can process one-hot vectors directly, they are inefficient because they are high-dimensional, sparse, and contain no information about the relationships between words. An embedding layer transforms these vectors into compact, dense representations that are easier for the model to learn from.

### **One-Hot Encoding:**

A one-hot vector represents each word by assigning a unique position in the vocabulary.

Example vocabulary:

Index	Word
0	hello
1	world
2	I
3	love
4	AI

Representation:

hello → [1, 0, 0, 0, 0]
world → [0, 1, 0, 0, 0]
I → [0, 0, 1, 0, 0]

Each word has exactly one element equal to 1, while the remaining values are 0.

### **Limitations of One-Hot Vectors:**
1. **High Dimensionality**

For a vocabulary containing thousands of words, each one-hot vector must have the same number of elements. Since most entries are zero, this representation wastes memory and increases computational cost.
If the vocabulary has 10,000 words, then the word "cat" is represented by a 10,000-dimensional one-hot vector, where only one value is 1 and the remaining 9,999 values are 0. This wastes memory.

Example:

Vocabulary: [cat, dog, apple, ...]
cat = [1, 0, 0, 0, ..., 0]

2. **No Semantic Information**

One-hot vectors only identify words and do not express any relationship between them. For example, dog and cat are represented as completely different vectors even though they are semantically related.
One-hot encoding treats all words as equally different.

Example:

dog = [1, 0, 0]
cat = [0, 1, 0]
apple = [0, 0, 1]

Although dog and cat are similar animals, their vectors do not reflect this similarity.

3. **Larger Weight Matrices**

Using one-hot vectors directly requires larger input weight matrices inside the RNN, increasing the number of parameters that must be learned. This results in slower training and higher memory usage.
Suppose the vocabulary size is 10,000 and the embedding size is 128.

The input weight matrix must be:

10,000 × 128

This means the model has to learn 1,280,000 parameters, increasing memory usage and training time.

In [2]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [3]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [4]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [5]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [6]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [7]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [8]:
PATH = r'eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['il se rapproche', 'he s coming closer']


15

### **Encoder:**

The encoder receives the input sentence one token at a time. Each word is first converted into an embedding vector before being processed by the Recurrent Neural Network (RNN). At every time step, the hidden state is updated to retain information from the current and previous words.

After the final input token has been processed, the encoder produces a final hidden state called the context vector, which summarizes the meaning of the entire input sentence.

The hidden state is computed as:

ht = f(xt,ht−1)

where,

xt = input token at time step t
ht−1 = previous hidden state
ht = updated hidden state
f = recurrent function (RNN)

In [9]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

### **Decoder:**
The decoder generates the output sentence one word at a time.

It begins with a special Start-of-Sequence (SOS) token and uses the encoder's context vector as its initial hidden state. At each step, the decoder predicts the next word in the sequence. The generated word is then fed back into the decoder until the End-of-Sequence (EOS) token is produced or the maximum sequence length is reached.


### Decoder During Inference:

During inference (testing), the target output is unknown. Hence, the decoder uses the word it predicted in the previous step as the input for the next step. This process continues until the **`<EOS>` (End-of-Sequence)** token is generated.

### **Why `topk(1)`?**

The decoder outputs a probability score for every word in the vocabulary.

```python
_, topi = decoder_output.topk(1)
```

The `topk(1)` function selects the word with the **highest probability**, and its index is passed as the next input to the decoder.

### **Why `detach()`?**

```python
decoder_input = topi.squeeze(-1).detach()
```

The `detach()` function separates the predicted token from the computation graph, preventing gradient calculations during inference. This reduces memory usage and improves execution speed.



### Encoder vs. Decoder:
1. Functional Comparison

|     Encoder                                |    Decoder                                         |
| ------------------------------------------ | ------------------------------------------------- |
| Accepts and encodes the input sentence.    | Produces the translated output sentence.          |
| Generates a context vector (hidden state). | Uses the context vector to predict the next word. |
| Processes the complete input sequence.     | Generates the output one token at a time.         |
| Does not produce translated words.         | Stops when the `<EOS>` token is predicted.        |



2. Code Comparison

|      Encoder                                                  |    Decoder                                                      |
| ------------------------------------------------------------- | --------------------------------------------------------------- |
| `output, hidden = self.rnn(embedded)`                         | `output, hidden = self.rnn(input, hidden)`                      |
| Processes the entire input sequence in a single forward pass. | Processes one token at each decoding step.                      |
| Sequence handling is performed internally by the RNN.         | The decoding loop is explicitly implemented using a `for` loop. |


In [10]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden

In [11]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [12]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [13]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [14]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [15]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [16]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [17]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [18]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 8s (- 5m 22s) (5 2%) 1.9824
0m 11s (- 3m 46s) (10 5%) 1.3064
0m 15s (- 3m 12s) (15 7%) 1.1025
0m 19s (- 2m 57s) (20 10%) 0.9509
0m 23s (- 2m 44s) (25 12%) 0.8240
0m 27s (- 2m 34s) (30 15%) 0.7149
0m 30s (- 2m 25s) (35 17%) 0.6205
0m 34s (- 2m 18s) (40 20%) 0.5362
0m 38s (- 2m 13s) (45 22%) 0.4591
0m 42s (- 2m 8s) (50 25%) 0.3923
0m 47s (- 2m 4s) (55 27%) 0.3383
0m 51s (- 2m 0s) (60 30%) 0.2931
0m 56s (- 1m 56s) (65 32%) 0.2553
1m 0s (- 1m 51s) (70 35%) 0.2188
1m 4s (- 1m 47s) (75 37%) 0.1923
1m 8s (- 1m 43s) (80 40%) 0.1715
1m 13s (- 1m 39s) (85 42%) 0.1514
1m 17s (- 1m 34s) (90 45%) 0.1374
1m 21s (- 1m 30s) (95 47%) 0.1216
1m 25s (- 1m 25s) (100 50%) 0.1103
1m 29s (- 1m 21s) (105 52%) 0.0990
1m 34s (- 1m 16s) (110 55%) 0.0959
1m 38s (- 1m 12s) (115 57%) 0.0891
1m 42s (- 1m 8s) (120 60%) 0.0810
1m 46s (- 1m 4s) (125 62%) 0.0797
1m 51s (- 0m 59s) (130 65%) 0.07

In [19]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> ce sont des melons
= they are melons
< they are melons <EOS>

> je me meurs
= i m dying
< you re very wise <EOS>

> je suis affole
= i m frantic
< you re all mad <EOS>

> je suis pret
= i m prepared
< you re too slow <EOS>

> tu es grande
= you are big
< he is my uncle <EOS>

> je suis celibataire
= i am single
< you re too skinny <EOS>

> nous perdons
= we re losing
< we re going now <EOS>

> vous etes sournois
= you re sneaky
< he is in pajamas <EOS>

> je suis un artiste
= i m an artist
< i am an artist <EOS>

> vous etes fort avisees
= you re very wise
< you re very wise <EOS>



### **Discussion:**

The Seq2Seq RNN Encoder–Decoder model was successfully implemented for machine translation. The encoder encoded the input sentence into a context vector, while the decoder generated the corresponding translated sequence. The gradual reduction in training loss demonstrated that the model effectively learned the relationship between the source and target languages. Teacher forcing improved the training process by accelerating convergence and reducing prediction errors. Although the model produced accurate results for short sentences, its performance was limited for longer and more complex sequences due to the fixed-length context vector.

### **Conclusion:**

The RNN Encoder-Decoder model was successfully implemented for sequence-to-sequence translation. The implementation included text preprocessing, vocabulary creation, model training, and translation generation. The model performed well on simple sentences, while advanced architectures such as LSTM, GRU, Attention, and Transformers can further improve translation accuracy. Overall, it provided a strong foundation in sequence-to-sequence learning and neural machine translation.